In [1]:
import sys
import pandas as pd
import numpy as np
import matplotlib
import seaborn as sns
import sklearn
import joblib
import fastapi

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("matplotlib:", matplotlib.__version__)
print("seaborn:", sns.__version__)
print("scikit-learn:", sklearn.__version__)
print("joblib:", joblib.__version__)
print("fastapi:", fastapi.__version__)

Python: 3.12.10
pandas: 3.1.0
numpy: 2.5.4
matplotlib: 3.11.2
seaborn: 0.13.2
scikit-learn: 1.9.1
joblib: 1.6.0
fastapi: 0.159.1


In [2]:
import pandas as pd

df = pd.read_csv('../datasets/raw/student-mat.csv', sep=';')

def risk_label(g3):
    if g3 < 10:
        return 'at_risk'
    elif g3 <= 12:
        return 'needs_review'
    else:
        return 'on_track'

df['risk_label'] = df['G3'].apply(risk_label)

print(df['risk_label'].value_counts())
print()
print(df['risk_label'].value_counts(normalize=True).round(3))

risk_label
needs_review    134
on_track        131
at_risk         130
Name: count, dtype: int64

risk_label
needs_review    0.339
on_track        0.332
at_risk         0.329
Name: proportion, dtype: float64


In [3]:
at_risk_df = df[df['risk_label'] == 'at_risk']
print(at_risk_df['G3'].value_counts().sort_index())

G3
0    38
4     1
5     7
6    15
7     9
8    32
9    28
Name: count, dtype: int64


In [4]:
print(df.isnull().sum().sum(), "missing values total")
print()
print(df['absences'].describe())

0 missing values total

count    395.000000
mean       5.708861
std        8.003096
min        0.000000
25%        0.000000
50%        4.000000
75%        8.000000
max       75.000000
Name: absences, dtype: float64


In [5]:
print(df.groupby('risk_label')['absences'].describe()[['count', 'mean', '50%', 'max']])

              count      mean  50%   max
risk_label                              
at_risk       130.0  6.761538  4.0  75.0
needs_review  134.0  5.716418  4.0  54.0
on_track      131.0  4.656489  3.0  24.0


In [6]:
from scipy.stats import kruskal

groups = [g['absences'].values for _, g in df.groupby('risk_label')]
stat, p = kruskal(*groups)
print("Kruskal-Wallis H:", round(stat, 3))
print("p-value:", round(p, 4))

Kruskal-Wallis H: 1.199
p-value: 0.5492


In [7]:
numeric_features = ['G1', 'G2', 'failures', 'studytime', 'goout', 'health', 'Dalc']
print(df.groupby('risk_label')[numeric_features].mean().round(2))

print()
print(pd.crosstab(df['risk_label'], df['higher'], normalize='index').round(2))

                 G1     G2  failures  studytime  goout  health  Dalc
risk_label                                                          
at_risk        7.76   6.82      0.69       1.95   3.40    3.68  1.55
needs_review  10.53  10.64      0.23       2.01   2.98    3.59  1.51
on_track      14.42  14.65      0.08       2.15   2.95    3.39  1.37

higher          no   yes
risk_label              
at_risk       0.10  0.90
needs_review  0.04  0.96
on_track      0.01  0.99


In [8]:
text_cols = df.select_dtypes(exclude='number').columns.tolist()
num_cols = df.select_dtypes(include='number').columns.tolist()

print("Text columns:", len(text_cols))
print(text_cols)
print()
print("Numeric columns:", len(num_cols))

Text columns: 18
['school', 'sex', 'address', 'famsize', 'Pstatus', 'Mjob', 'Fjob', 'reason', 'guardian', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic', 'risk_label']

Numeric columns: 16


In [9]:
from sklearn.model_selection import train_test_split

y = df['risk_label']

# Feature sets
features_a = [c for c in df.columns if c not in ['G2', 'G3', 'risk_label']]  # early flagging
features_b = [c for c in df.columns if c not in ['G3', 'risk_label']]         # after period 2

# Split row positions once, so both models see the same students
train_idx, test_idx = train_test_split(
    df.index, test_size=0.2, stratify=y, random_state=42
)

print("Model A features:", len(features_a))
print("Model B features:", len(features_b))
print("Train rows:", len(train_idx), "| Test rows:", len(test_idx))
print()
print("Train class proportions:")
print(y.loc[train_idx].value_counts(normalize=True).round(3))
print()
print("Test class proportions:")
print(y.loc[test_idx].value_counts(normalize=True).round(3))

Model A features: 31
Model B features: 32
Train rows: 316 | Test rows: 79

Train class proportions:
risk_label
needs_review    0.339
on_track        0.332
at_risk         0.329
Name: proportion, dtype: float64

Test class proportions:
risk_label
needs_review    0.342
on_track        0.329
at_risk         0.329
Name: proportion, dtype: float64


In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

def build_preprocessor(features):
    part = df[features]
    cat_cols = part.select_dtypes(exclude='number').columns.tolist()
    num_cols = part.select_dtypes(include='number').columns.tolist()
    return ColumnTransformer([
        ('cat', OneHotEncoder(drop='if_binary', handle_unknown='ignore'), cat_cols),
        ('num', StandardScaler(), num_cols),
    ])

pre_a = build_preprocessor(features_a)

# Learn the rules from training rows only, then apply them to both sets
X_train_a = pre_a.fit_transform(df.loc[train_idx, features_a])
X_test_a = pre_a.transform(df.loc[test_idx, features_a])

print("Training matrix:", X_train_a.shape)
print("Test matrix:", X_test_a.shape)
print()
print(pre_a.get_feature_names_out()[:10])

Training matrix: (316, 44)
Test matrix: (79, 44)

['cat__school_MS' 'cat__sex_M' 'cat__address_U' 'cat__famsize_LE3'
 'cat__Pstatus_T' 'cat__Mjob_at_home' 'cat__Mjob_health' 'cat__Mjob_other'
 'cat__Mjob_services' 'cat__Mjob_teacher']


In [11]:
pre_b = build_preprocessor(features_b)

X_train_b = pre_b.fit_transform(df.loc[train_idx, features_b])
X_test_b = pre_b.transform(df.loc[test_idx, features_b])

print("Training matrix:", X_train_b.shape)
print("Test matrix:", X_test_b.shape)

Training matrix: (316, 45)
Test matrix: (79, 45)


In [12]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

y_train = df.loc[train_idx, 'risk_label']
y_test = df.loc[test_idx, 'risk_label']

experiments = {
    'Baseline (always most common)': (DummyClassifier(strategy='most_frequent'), X_train_a, X_test_a),
    'Logistic regression, Model A': (LogisticRegression(max_iter=1000, class_weight='balanced'), X_train_a, X_test_a),
    'Logistic regression, Model B': (LogisticRegression(max_iter=1000, class_weight='balanced'), X_train_b, X_test_b),
}

for name, (model, X_tr, X_te) in experiments.items():
    model.fit(X_tr, y_train)
    pred = model.predict(X_te)
    acc = accuracy_score(y_test, pred)
    macro_f1 = f1_score(y_test, pred, average='macro')
    print(f"{name}: accuracy={acc:.3f}, macro F1={macro_f1:.3f}")

Baseline (always most common): accuracy=0.342, macro F1=0.170
Logistic regression, Model A: accuracy=0.671, macro F1=0.670
Logistic regression, Model B: accuracy=0.785, macro F1=0.783


In [13]:
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ['accuracy', 'f1_macro']

pipe_a = make_pipeline(build_preprocessor(features_a),
                       LogisticRegression(max_iter=1000, class_weight='balanced'))
pipe_b = make_pipeline(build_preprocessor(features_b),
                       LogisticRegression(max_iter=1000, class_weight='balanced'))

res_a = cross_validate(pipe_a, df.loc[train_idx, features_a], y_train, cv=cv, scoring=scoring)
res_b = cross_validate(pipe_b, df.loc[train_idx, features_b], y_train, cv=cv, scoring=scoring)

for name, res in [('Model A', res_a), ('Model B', res_b)]:
    acc = res['test_accuracy']
    f1 = res['test_f1_macro']
    print(f"{name}: accuracy={acc.mean():.3f} (±{acc.std():.3f}), "
          f"macro F1={f1.mean():.3f} (±{f1.std():.3f})")

Model A: accuracy=0.693 (±0.077), macro F1=0.690 (±0.083)
Model B: accuracy=0.810 (±0.041), macro F1=0.811 (±0.042)


In [14]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import classification_report, confusion_matrix

pred_a = cross_val_predict(pipe_a, df.loc[train_idx, features_a], y_train, cv=cv)

print(classification_report(y_train, pred_a, digits=3))
print(confusion_matrix(y_train, pred_a, labels=['at_risk', 'needs_review', 'on_track']))

              precision    recall  f1-score   support

     at_risk      0.727     0.769     0.748       104
needs_review      0.563     0.542     0.552       107
    on_track      0.786     0.771     0.779       105

    accuracy                          0.693       316
   macro avg      0.692     0.694     0.693       316
weighted avg      0.691     0.693     0.692       316

[[80 22  2]
 [29 58 20]
 [ 1 23 81]]


In [15]:
import numpy as np

flagged_true = y_train.isin(['at_risk', 'needs_review']).values
flagged_pred = np.isin(pred_a, ['at_risk', 'needs_review'])

tp = np.sum(flagged_true & flagged_pred)
recall = tp / flagged_true.sum()
precision = tp / flagged_pred.sum()
print(f"Two-tier flag: recall={recall:.3f}, precision={precision:.3f}")
print(f"Students flagged: {flagged_pred.sum()} of {len(flagged_pred)}")

Two-tier flag: recall=0.896, precision=0.887
Students flagged: 213 of 316


In [16]:
from sklearn.metrics import classification_report, confusion_matrix

final_a = make_pipeline(build_preprocessor(features_a),
                        LogisticRegression(max_iter=1000, class_weight='balanced'))
final_a.fit(df.loc[train_idx, features_a], y_train)

test_pred = final_a.predict(df.loc[test_idx, features_a])

print(classification_report(y_test, test_pred, digits=3))
print(confusion_matrix(y_test, test_pred, labels=['at_risk', 'needs_review', 'on_track']))

flagged_true = y_test.isin(['at_risk', 'needs_review']).values
flagged_pred = np.isin(test_pred, ['at_risk', 'needs_review'])
tp = np.sum(flagged_true & flagged_pred)
print(f"Two-tier flag: recall={tp / flagged_true.sum():.3f}, "
      f"precision={tp / flagged_pred.sum():.3f}")
print(f"Students flagged: {flagged_pred.sum()} of {len(flagged_pred)}")

              precision    recall  f1-score   support

     at_risk      0.769     0.769     0.769        26
needs_review      0.520     0.481     0.500        27
    on_track      0.714     0.769     0.741        26

    accuracy                          0.671        79
   macro avg      0.668     0.673     0.670        79
weighted avg      0.666     0.671     0.668        79

[[20  6  0]
 [ 6 13  8]
 [ 0  6 20]]
Two-tier flag: recall=0.849, precision=0.882
Students flagged: 51 of 79


In [17]:
import os

joblib.dump(final_a, '../models/model_a_pipeline.joblib')
joblib.dump(features_a, '../models/model_a_features.joblib')

reloaded = joblib.load('../models/model_a_pipeline.joblib')
check = reloaded.predict(df.loc[test_idx, features_a])

print("Reloaded predictions match:", np.array_equal(check, test_pred))
print("Pipeline file size (KB):", round(os.path.getsize('../models/model_a_pipeline.joblib') / 1024, 1))
print("Feature count:", len(joblib.load('../models/model_a_features.joblib')))

Reloaded predictions match: True
Pipeline file size (KB): 9.1
Feature count: 31


In [18]:
import sys
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, f1_score

sys.path.insert(0, '../src')
from preprocessing import load_data, risk_label, TARGET

LABELS = ['at_risk', 'needs_review', 'on_track']
FLAG = ['at_risk', 'needs_review']

# Success criterion, written before running:
MIN_FLAG_RECALL = 0.80   # two-tier recall must stay at or above this
# and macro F1 must beat the always-most-common baseline on the Portuguese data

KEY = ['school', 'sex', 'age', 'address', 'famsize', 'Pstatus', 'Medu', 'Fedu',
       'Mjob', 'Fjob', 'reason', 'nursery', 'internet']

math = load_data()
por = pd.read_csv('../datasets/raw/student-por.csv', sep=';')
por[TARGET] = por['G3'].apply(risk_label)

pipeline = joblib.load('../models/model_a_pipeline.joblib')
features = joblib.load('../models/model_a_features.joblib')

# Remove Portuguese students who also appear in the Math file
match = por.merge(math[KEY].drop_duplicates(), on=KEY, how='left', indicator=True)
por_clean = por[(match['_merge'] == 'left_only').values].reset_index(drop=True)
print(f"Portuguese rows: {len(por)} | overlap removed: {len(por) - len(por_clean)} | kept: {len(por_clean)}")

# Check for categories the model never saw in training
for c in math[features].select_dtypes(exclude='number').columns:
    unseen = set(por_clean[c].unique()) - set(math[c].unique())
    if unseen:
        print(f"WARNING unseen values in {c}: {unseen}")

X, y = por_clean[features], por_clean[TARGET]
pred = pipeline.predict(X)

print("\nClass proportions (Portuguese, after overlap removal):")
print(y.value_counts(normalize=True).round(3))
print()
print(classification_report(y, pred, labels=LABELS, digits=3, zero_division=0))
print(confusion_matrix(y, pred, labels=LABELS))

macro_f1 = f1_score(y, pred, labels=LABELS, average='macro', zero_division=0)
baseline_pred = np.repeat(y.value_counts().idxmax(), len(y))
baseline_f1 = f1_score(y, baseline_pred, labels=LABELS, average='macro', zero_division=0)

flagged_true = y.isin(FLAG).values
flagged_pred = np.isin(pred, FLAG)
tp = np.sum(flagged_true & flagged_pred)
flag_recall = tp / flagged_true.sum()
flag_precision = tp / flagged_pred.sum()

print(f"\nMacro F1: {macro_f1:.3f} (baseline: {baseline_f1:.3f})")
print(f"Two-tier flag: recall={flag_recall:.3f}, precision={flag_precision:.3f}")
print(f"Students flagged: {flagged_pred.sum()} of {len(flagged_pred)}")

passed = flag_recall >= MIN_FLAG_RECALL and macro_f1 > baseline_f1
print("\nVERDICT:", "criterion met" if passed else "criterion not met")

Portuguese rows: 649 | overlap removed: 374 | kept: 275

Class proportions (Portuguese, after overlap removal):
risk_label
needs_review    0.436
on_track        0.309
at_risk         0.255
Name: proportion, dtype: float64

              precision    recall  f1-score   support

     at_risk      0.563     0.700     0.624        70
needs_review      0.598     0.583     0.591       120
    on_track      0.817     0.682     0.744        85

    accuracy                          0.644       275
   macro avg      0.659     0.655     0.653       275
weighted avg      0.657     0.644     0.646       275

[[49 21  0]
 [37 70 13]
 [ 1 26 58]]

Macro F1: 0.653 (baseline: 0.203)
Two-tier flag: recall=0.932, precision=0.868
Students flagged: 204 of 275

VERDICT: criterion met


In [19]:
print("Math key dtypes:")
print(math[KEY].dtypes.to_dict())
print("\nPortuguese key dtypes:")
print(por[KEY].dtypes.to_dict())

mat_keys = set(map(tuple, math[KEY].values))
por_keys = list(map(tuple, por[KEY].values))
print("\nExact 13-key matches:", sum(k in mat_keys for k in por_keys))

# How many matches survive if we drop the most specific keys?
for n in [13, 10, 8, 5]:
    keys = KEY[:n]
    mk = set(map(tuple, math[keys].values))
    pk = list(map(tuple, por[keys].values))
    print(f"Using first {n} keys: {sum(k in mk for k in pk)} Portuguese rows match")

Math key dtypes:
{'school': <StringDtype(storage='python', na_value=nan)>, 'sex': <StringDtype(storage='python', na_value=nan)>, 'age': dtype('int64'), 'address': <StringDtype(storage='python', na_value=nan)>, 'famsize': <StringDtype(storage='python', na_value=nan)>, 'Pstatus': <StringDtype(storage='python', na_value=nan)>, 'Medu': dtype('int64'), 'Fedu': dtype('int64'), 'Mjob': <StringDtype(storage='python', na_value=nan)>, 'Fjob': <StringDtype(storage='python', na_value=nan)>, 'reason': <StringDtype(storage='python', na_value=nan)>, 'nursery': <StringDtype(storage='python', na_value=nan)>, 'internet': <StringDtype(storage='python', na_value=nan)>}

Portuguese key dtypes:
{'school': <StringDtype(storage='python', na_value=nan)>, 'sex': <StringDtype(storage='python', na_value=nan)>, 'age': dtype('int64'), 'address': <StringDtype(storage='python', na_value=nan)>, 'famsize': <StringDtype(storage='python', na_value=nan)>, 'Pstatus': <StringDtype(storage='python', na_value=nan)>, 'Medu': d

In [20]:
print("Math rows:", len(math), "| Portuguese rows:", len(por))
print("Math header matches Portuguese header:", list(math.columns[:33]) == list(por.columns[:33]))

# Same inner join as UCI's R script
joined = math[KEY].merge(por[KEY], on=KEY, how='inner')
print("Inner-join rows (UCI-style count):", len(joined))

# Unique Portuguese students matched
unique_por = por.merge(math[KEY].drop_duplicates(), on=KEY, how='inner')
print("Unique Portuguese rows matched:", len(unique_por))

print("\nFirst 3 Portuguese rows:")
print(por[['school', 'sex', 'age', 'G1', 'G2', 'G3']].head(3))

Math rows: 395 | Portuguese rows: 649
Math header matches Portuguese header: True
Inner-join rows (UCI-style count): 382
Unique Portuguese rows matched: 374

First 3 Portuguese rows:
  school sex  age  G1  G2  G3
0     GP   F   18   0  11  11
1     GP   F   17   9  11  11
2     GP   F   15  12  13  12


In [21]:
zeros = df[df['G3'] == 0]
print(zeros[['G1', 'G2', 'G3']].describe().round(2))
print("Zeros with G2 = 0:", (zeros['G2'] == 0).sum(), "of", len(zeros))

          G1     G2    G3
count  38.00  38.00  38.0
mean    7.53   4.66   0.0
std     1.81   3.70   0.0
min     4.00   0.00   0.0
25%     6.00   0.00   0.0
50%     7.00   5.00   0.0
75%     9.00   8.00   0.0
max    12.00  10.00   0.0
Zeros with G2 = 0: 13 of 38


In [22]:
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

# Remove the 38 students with a final grade of 0
df_s = df[df['G3'] > 0].reset_index(drop=True)
y_s = df_s['risk_label']
print("Students kept:", len(df_s), "| removed:", len(df) - len(df_s))
print(y_s.value_counts())
print()

feat_a_s = [c for c in df_s.columns if c not in ['G2', 'G3', 'risk_label']]
feat_b_s = [c for c in df_s.columns if c not in ['G3', 'risk_label']]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ['accuracy', 'f1_macro']

for name, feats in [('Model A', feat_a_s), ('Model B', feat_b_s)]:
    pipe = make_pipeline(build_preprocessor(feats),
                         LogisticRegression(max_iter=1000, class_weight='balanced'))
    res = cross_validate(pipe, df_s[feats], y_s, cv=cv, scoring=scoring)
    acc, f1 = res['test_accuracy'], res['test_f1_macro']
    print(f"{name} CV (no G3 = 0): accuracy={acc.mean():.3f} (±{acc.std():.3f}), "
          f"macro F1={f1.mean():.3f} (±{f1.std():.3f})")

Students kept: 357 | removed: 38
risk_label
needs_review    134
on_track        131
at_risk          92
Name: count, dtype: int64

Model A CV (no G3 = 0): accuracy=0.658 (±0.067), macro F1=0.659 (±0.070)
Model B CV (no G3 = 0): accuracy=0.810 (±0.063), macro F1=0.812 (±0.066)


In [23]:
print("Remaining students with G2 = 0:", (df_s['G2'] == 0).sum())

Remaining students with G2 = 0: 0


In [24]:
import numpy as np

flag_true = y_train.isin(['at_risk', 'needs_review']).values
flag_pred = np.isin(pred_a, ['at_risk', 'needs_review'])

sub = df.loc[train_idx].copy()
sub['flag_true'] = flag_true
sub['flag_pred'] = flag_pred

for col in ['sex', 'school', 'address']:
    print(f"\n--- by {col} ---")
    for group, d in sub.groupby(col):
        hits = (d['flag_true'] & d['flag_pred']).sum()
        recall = hits / d['flag_true'].sum()
        precision = hits / max(d['flag_pred'].sum(), 1)
        print(f"{group}: n={len(d)}, "
              f"share needing attention={d['flag_true'].mean():.2f}, "
              f"flagged={d['flag_pred'].mean():.2f}, "
              f"recall={recall:.2f}, precision={precision:.2f}")


--- by sex ---
F: n=163, share needing attention=0.68, flagged=0.69, recall=0.90, precision=0.88
M: n=153, share needing attention=0.65, flagged=0.65, recall=0.89, precision=0.89

--- by school ---
GP: n=279, share needing attention=0.66, flagged=0.67, recall=0.90, precision=0.89
MS: n=37, share needing attention=0.73, flagged=0.73, recall=0.89, precision=0.89

--- by address ---
R: n=73, share needing attention=0.79, flagged=0.81, recall=0.91, precision=0.90
U: n=243, share needing attention=0.63, flagged=0.63, recall=0.89, precision=0.88


In [25]:
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss
from sklearn.model_selection import cross_val_predict

proba_cv = cross_val_predict(pipe_a, df.loc[train_idx, features_a], y_train,
                             cv=cv, method='predict_proba')

classes = sorted(y_train.unique())
flag_prob = (proba_cv[:, classes.index('at_risk')] +
             proba_cv[:, classes.index('needs_review')])
flag_obs = y_train.isin(['at_risk', 'needs_review']).astype(int).values

frac_pos, mean_pred = calibration_curve(flag_obs, flag_prob, n_bins=5, strategy='quantile')

print("Predicted flag probability -> observed share needing attention:")
for p, o in zip(mean_pred, frac_pos):
    print(f"  predicted {p:.2f} -> observed {o:.2f}")

base = flag_obs.mean()
print(f"\nBrier score (model): {brier_score_loss(flag_obs, flag_prob):.3f}")
print(f"Brier score (always predict base rate {base:.2f}): {base * (1 - base):.3f}")

Predicted flag probability -> observed share needing attention:
  predicted 0.04 -> observed 0.11
  predicted 0.46 -> observed 0.49
  predicted 0.89 -> observed 0.78
  predicted 0.99 -> observed 0.97
  predicted 1.00 -> observed 1.00

Brier score (model): 0.105
Brier score (always predict base rate 0.67): 0.222


In [26]:
cuts = [0.3, 0.8]
band_names = np.array(['low', 'moderate', 'high'])
bands = band_names[np.digitize(flag_prob, cuts)]

check = pd.DataFrame({'band': bands, 'needs_attention': flag_obs})
print(check.groupby('band')['needs_attention'].agg(['count', 'mean']).round(2))

          count  mean
band                 
high        179  0.94
low          79  0.18
moderate     58  0.50
